# 03. Explain with SHAP and Gemini

Moringa Masterclass: Machine Learning End to End

This is part 3 of 3. We take individual predictions from the model, compute SHAP values to see which features drove each prediction, and then use Gemini to turn those numbers into a plain-language explanation someone without an ML background could read.

We also walk through a case where the LLM explanation needs correcting against the real SHAP values. That is the main takeaway of this notebook: treat LLM explanations as a communication layer on top of a real explanation method, not as the explanation method itself.

In [ ]:
%pip install -q scikit-learn pandas numpy shap google-genai

## 1. Get a Gemini API key

1. Go to https://aistudio.google.com/apikey
2. Sign in with a Google account
3. Click Create API key
4. Copy it and paste it below

This is free at the tier we are using today, and takes under a minute. Do not commit your key to GitHub. In Colab, use the Secrets panel (the key icon in the left sidebar) instead of pasting it directly into a cell.

In [ ]:
import os

# In Colab, prefer: from google.colab import userdata; GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "PASTE_YOUR_KEY_HERE")

from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)
GEMINI_MODEL_NAME = "gemini-2.5-flash"


## 2. Rebuild the model

For the live session we retrain quickly rather than reload from the MLflow registry, so this notebook can also run standalone. In your own projects you would normally load the registered model directly with `mlflow.pyfunc.load_model`.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("data/Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)

target = "Churn"
X = df.drop(columns=["customerID", target])
y = (df[target] == "Yes").astype(int)

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)),
])
model.fit(X_train, y_train)

## 3. Compute SHAP values

SHAP tells us, for a single prediction, how much each feature pushed the prediction up or down relative to the average. We run it on the fitted classifier, using the already-preprocessed features so the feature names line up with what the model actually saw.

In [ ]:
import shap
import numpy as np

# Transform the test set through the same preprocessing the model uses internally,
# so SHAP sees the actual features the classifier was trained on.
X_test_transformed = model.named_steps["preprocessor"].transform(X_test)
feature_names = model.named_steps["preprocessor"].get_feature_names_out()

if hasattr(X_test_transformed, "toarray"):
    X_test_transformed = X_test_transformed.toarray()

explainer = shap.TreeExplainer(model.named_steps["classifier"])
shap_values = explainer.shap_values(X_test_transformed)

# shap_values has one array per class for this classifier; we want the "churn" class (index 1)
shap_values_churn = shap_values[:, :, 1] if np.ndim(shap_values) == 3 else shap_values[1]

In [ ]:
shap.summary_plot(
    shap_values_churn, X_test_transformed, feature_names=feature_names, show=True
)

That summary plot is the kind of thing that is genuinely useful to a data scientist and genuinely unreadable to most stakeholders. That gap is exactly what the next step addresses.

## 4. Explain a single prediction

Pick one customer, get their SHAP values, and see the raw numbers before we hand anything to an LLM.

In [ ]:
def explain_row(row_index, top_n=6):
    row_shap = shap_values_churn[row_index]
    contributions = pd.Series(row_shap, index=feature_names).sort_values(key=abs, ascending=False)
    top_contributions = contributions.head(top_n)

    prediction_proba = model.predict_proba(X_test.iloc[[row_index]])[0][1]

    return {
        "customer_index": row_index,
        "predicted_churn_probability": round(float(prediction_proba), 3),
        "top_features": [
            {"feature": name, "shap_value": round(float(value), 3)}
            for name, value in top_contributions.items()
        ],
    }

example = explain_row(0)
example

## 5. Ask Gemini to narrate it

We send the raw SHAP contributions, not a vague summary, so Gemini has real numbers to ground its explanation in. The prompt explicitly asks it to only use the numbers given.

In [ ]:
def build_prompt(explanation):
    features_text = "\n".join(
        f"- {f['feature']}: SHAP value {f['shap_value']} "
        f"({'pushes toward churn' if f['shap_value'] > 0 else 'pushes away from churn'})"
        for f in explanation["top_features"]
    )
    return f'''A machine learning model predicts this customer has a
{explanation['predicted_churn_probability'] * 100:.0f}% probability of churning.

Here are the top features driving that prediction, from SHAP analysis:
{features_text}

Write a short, plain-language explanation (3 to 4 sentences) a non-technical
customer support manager could read and act on. Only use the information given
above. Do not invent additional reasons.'''

prompt = build_prompt(example)
print(prompt)

In [ ]:
response = client.models.generate_content(model=GEMINI_MODEL_NAME, contents=prompt)
print(response.text)

## 6. Where LLM explanations go wrong, and how to catch it

LLMs are good at turning numbers into fluent sentences. They are not reliably good at getting the direction or magnitude of an effect exactly right, especially with encoded categorical features (for example, a one-hot column like `cat__Contract_Month-to-month`). Always check the generated explanation against the actual SHAP values before trusting it.

Below, we deliberately look at a case with a less intuitive feature (an encoded categorical variable) and check the LLM's explanation line by line against the numbers.

In [ ]:
tricky_example = explain_row(3)
tricky_prompt = build_prompt(tricky_example)
tricky_response = client.models.generate_content(model=GEMINI_MODEL_NAME, contents=tricky_prompt)

print("SHAP values Gemini was given:")
for f in tricky_example["top_features"]:
    print(" ", f)

print("\nGemini's explanation:")
print(tricky_response.text)

print("\nCheck: does every claim in the explanation above match a feature and direction in the list?")
print("If Gemini describes a feature that is not in the list, or gets the direction backwards,")
print("that is the failure mode to point out live: the explanation reads well but is not grounded.")

## Recap: the full blueprint

Train with Scikit-Learn, track with MLflow, explain with SHAP and Gemini. The same dataset and the same model ran through all three notebooks. That is the repeatable part: this structure applies to almost any tabular ML problem, not just churn.

Thank you for attending. If you want to go deeper into applied data science and MLOps, that is exactly what Moringa's programs cover.